In [ ]:
import pandas as pd
import re

llm = pd.read_csv('LLM_Sheet - Sheet1.csv')
llm = llm.dropna(subset=['text'])
s1  = pd.read_csv('Stage_1.csv')
s2  = pd.read_csv('Stage_2-1.csv')
s22 = pd.read_csv('Stage_2-2.csv')

In [2]:
# Stage 1 (latent): question columns only
s1_questions = [c for c in s1.columns if c not in ['Timestamp', 'Participant ID', 'Email Id', 'Attention Check']]

# Stage 2 (recent): question columns only
s2_questions = [c for c in s2.columns if c not in ['Timestamp', 'Participant ID']]

# Stage 2-2 (new + all 36): columns with believability ratings
s22_questions = [c for c in s22.columns if 'Believability' in c]
# Strip full suffix to get raw headline text
s22_texts = [re.split(r'\n\n?Believability\n', c)[0].strip() for c in s22_questions]

In [3]:
# Map each headline to its primary stage based on condition
records = []
for _, row in llm.iterrows():
    text      = row['text'].strip()
    condition = row['condition']
    matched_stage, matched_col = None, None

    if condition == 'latent' and text in s1_questions:
        matched_stage, matched_col = 'Stage 1', text
    elif condition == 'recent' and text in s2_questions:
        matched_stage, matched_col = 'Stage 2', text
    elif condition == 'new':
        for raw, col in zip(s22_texts, s22_questions):
            if text == raw:
                matched_stage, matched_col = 'Stage 2-2', col
                break

    records.append({'id': row['id'], 'text': text, 'condition': condition,
                    'stage': matched_stage, 'column_header': matched_col})

mapping_df = pd.DataFrame(records)
print(f"Total: {len(mapping_df)} | Matched: {mapping_df['stage'].notna().sum()} | Unmatched: {mapping_df['stage'].isna().sum()}")
mapping_df

Total: 36 | Matched: 36 | Unmatched: 0


,id,text,condition,stage,column_header
0,458.0,Soaking your feet in warm mustard oil pulls a ...,latent,Stage 1,Soaking your feet in warm mustard oil pulls a ...
1,492.0,Applying fresh milk directly to your face remo...,latent,Stage 1,Applying fresh milk directly to your face remo...
2,151.0,Beware: a terrifying new WhatsApp forward will...,recent,Stage 2,Beware: a terrifying new WhatsApp forward will...
3,482.0,Stop drinking tap water without boiling it imm...,latent,Stage 1,Stop drinking tap water without boiling it imm...
4,328.0,Urgent message: forward this to ten groups rig...,new,Stage 2-2,Urgent message: forward this to ten groups rig...
5,63.0,Act fast and delete this common navigation app...,recent,Stage 2,Act fast and delete this common navigation app...
6,112.0,Prominent doctors warn that drinking cold drin...,new,Stage 2-2,Prominent doctors warn that drinking cold drin...
7,141.0,Updating your phone software drains your batte...,recent,Stage 2,Updating your phone software drains your batte...
8,174.0,Eating an apple on an empty stomach prevents t...,recent,Stage 2,Eating an apple on an empty stomach prevents t...
9,126.0,Walking barefoot on morning grass improves you...,recent,Stage 2,Walking barefoot on morning grass improves you...


In [4]:
# Verify one-to-one mapping (primary stage)
print(mapping_df.groupby('stage')['id'].count())
assert mapping_df['stage'].notna().all(), f"Unmatched:\n{mapping_df[mapping_df['stage'].isna()]}"
print("\nOne-to-one mapping verified successfully.\n")

# Secondary check: all 36 headlines must exist in Stage 2-2
missing_in_s22 = [t for t in mapping_df['text'] if t not in s22_texts]
print(f"Headlines present in Stage 2-2: {len(mapping_df) - len(missing_in_s22)}/36")
if missing_in_s22:
    print(f"Missing from Stage 2-2: {missing_in_s22}")
else:
    print("All 36 headlines confirmed in Stage 2-2.")

stage
Stage 1      12
Stage 2      12
Stage 2-2    12
Name: id, dtype: int64

One-to-one mapping verified successfully.

Headlines present in Stage 2-2: 36/36
All 36 headlines confirmed in Stage 2-2.


In [5]:
# Check participant completion across all three stages
s1_ids  = set(s1['Participant ID'])
s2_ids  = set(s2['Participant ID'])
s22_ids = set(s22['Participation Id'])

all_ids = s1_ids | s2_ids | s22_ids

completion = pd.DataFrame({
    'participant_id': sorted(all_ids),
    'Stage 1':   [pid in s1_ids  for pid in sorted(all_ids)],
    'Stage 2':   [pid in s2_ids  for pid in sorted(all_ids)],
    'Stage 2-2': [pid in s22_ids for pid in sorted(all_ids)],
})
completion['all_stages_done'] = completion[['Stage 1', 'Stage 2', 'Stage 2-2']].all(axis=1)

print(f"Total unique participants : {len(all_ids)}")
print(f"Completed all 3 stages   : {completion['all_stages_done'].sum()}")
print(f"Missing at least 1 stage : {(~completion['all_stages_done']).sum()}\n")
completion

Total unique participants : 61
Completed all 3 stages   : 56
Missing at least 1 stage : 5



,participant_id,Stage 1,Stage 2,Stage 2-2,all_stages_done
0,14,True,True,True,True
1,15,True,True,True,True
2,16,True,True,True,True
3,17,True,True,True,True
4,18,True,True,True,True
...,...,...,...,...,...
56,70,True,True,True,True
57,71,True,True,True,True
58,72,True,True,True,True
59,73,True,False,False,False


In [6]:
# Verify attention check in Stage 1: all participants should have 'Yes'
attn = s1[['Participant ID', 'Attention Check']].copy()
attn['passed'] = attn['Attention Check'].str.strip().str.lower() == 'yes'

print(f"Passed attention check : {attn['passed'].sum()}/{len(attn)}")
failed = attn[~attn['passed']]
if failed.empty:
    print("All participants passed the attention check.")
else:
    print("Failed attention check:")
    print(failed)
attn

Passed attention check : 54/61
Failed attention check:
    Participant ID Attention Check  passed
0               14              No   False
3               17              No   False
13              26              No   False
28              42             NaN   False
34              48              No   False
39              53              No   False
52              66              No   False


,Participant ID,Attention Check,passed
0,14,No,False
1,15,Yes,True
2,16,Yes,True
3,17,No,False
4,18,Yes,True
...,...,...,...
56,70,Yes,True
57,71,Yes,True
58,72,Yes,True
59,73,Yes,True


In [7]:
# Check if all participants answered Yes to all questions in Stage 1 and Stage 2
for stage_name, df, q_cols in [('Stage 1', s1, s1_questions), ('Stage 2', s2, s2_questions)]:
    all_yes = df[q_cols].apply(lambda col: col.str.strip().str.lower() == 'yes')
    df_check = all_yes.all(axis=1).rename('all_yes')
    pid_col = 'Participant ID'
    result = pd.concat([df[pid_col], df_check], axis=1)
    failed = result[~result['all_yes']]
    print(f"{stage_name}: {df_check.sum()}/{len(df)} participants answered Yes to all questions.")
    if not failed.empty:
        print(f"  Participants with at least one non-Yes response:")
        for _, row in failed.iterrows():
            pid = row[pid_col]
            non_yes = df.loc[df[pid_col] == pid, q_cols].iloc[0]
            non_yes = non_yes[non_yes.str.strip().str.lower() != 'yes']
            print(f"    Participant {pid}: {non_yes.to_dict()}")
    print()

Stage 1: 60/61 participants answered Yes to all questions.
  Participants with at least one non-Yes response:
    Participant 14: {'Soaking your feet in warm mustard oil pulls a fever down because the heat from the oil forces the hot blood to move away from your head.': 'No', 'Consuming a small bowl of roasted chana for breakfast drastically improves your nighttime sleep.': 'No', 'Emergency medical warning: a mysterious new skin allergy is sending completely healthy children to the hospital across all major towns this week.': 'No'}

Stage 2: 56/56 participants answered Yes to all questions.



In [8]:
# Summary: all flagged participant IDs across all criteria (printed once each)

# 1. Missed at least one stage
missed_stage = set(completion.loc[~completion['all_stages_done'], 'participant_id'])

# 2. Failed attention check in Stage 1
failed_attn = set(attn.loc[~attn['passed'], 'Participant ID'])

# 3. Did not answer Yes to all questions in Stage 1 or Stage 2
s1_not_all_yes = set(s1.loc[~s1[s1_questions].apply(lambda col: col.str.strip().str.lower() == 'yes').all(axis=1), 'Participant ID'])
s2_not_all_yes = set(s2.loc[~s2[s2_questions].apply(lambda col: col.str.strip().str.lower() == 'yes').all(axis=1), 'Participant ID'])
not_all_yes = s1_not_all_yes | s2_not_all_yes

flagged = missed_stage | failed_attn | not_all_yes

print(f"Missed at least 1 stage     : {sorted(missed_stage)}")
print(f"Failed attention check       : {sorted(failed_attn)}")
print(f"Not all Yes in Stage 1 or 2  : {sorted(not_all_yes)}")
print(f"\nAll flagged participant IDs  : {sorted(flagged)}")

Missed at least 1 stage     : [26, 49, 51, 59, 73]
Failed attention check       : [14, 17, 26, 42, 48, 53, 66]
Not all Yes in Stage 1 or 2  : [14]

All flagged participant IDs  : [14, 17, 26, 42, 48, 49, 51, 53, 59, 66, 73]


In [ ]:
# Write Stage 2-2 without flagged participants
s22_clean = s22[~s22['Participation Id'].isin(flagged)]
s22_clean.to_csv('Cleaned.csv', index=False)
print(f"Original participants : {s22['Participation Id'].nunique()}")
print(f"Removed               : {len(flagged & s22_ids)}")
print(f"Remaining participants: {s22_clean['Participation Id'].nunique()}")
print("Saved to Cleaned.csv")

Original participants : 56
Removed               : 6
Remaining participants: 50
Saved to Stage 2-2 (Cleaned).csv
